# Day 32 — Data Manipulation with Pandas
**Week 6: Pandas & Data Visualization**
**Topics covered:** `groupby()`, `agg()`, `merge()`, `concat()`, `pivot_table()`, handling missing values (`fillna()`, `dropna()`), `apply()`, `map()`, renaming columns, dropping columns/rows, `reset_index()`
 
---
 
## Dataset
**Name:** `World Happiness Report 2024`
**URL:** `https://www.kaggle.com/datasets/jainaru/world-happiness-report-2024-yearly-updated`
**Download:**
```python
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jainaru/world-happiness-report-2024-yearly-updated")

print("Path to dataset files:", path)
```
**Key Columns:** `Country name`, `year`, `Life Ladder`, `Log GDP per capita`, `Social support`, `Healthy life expectancy at birth`, `Freedom to make life choices`, `Generosity`, `Perceptions of corruption`, `Regional indicator`
 
---

**Q1.** Load the dataset. Drop the columns `Positive affect` and `Negative affect` if they exist. Print the remaining column names. Then rename `Life Ladder` to `Ladder Score` and `Country name` to `Country`.
 

In [1]:
# Q1: Load Titanic dataset and display first 5 and last 3 rows

import pandas as pd
import kagglehub, os

# Get path to downloaded dataset
path = kagglehub.dataset_download("jainaru/world-happiness-report-2024-yearly-updated")

# Find the CSV file inside the downloaded folder
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file))

print("=== First 5 rows ===")
print(df.head())

print("\n=== Last 3 rows ===")
print(df.tail(3))

100%|███████████████████████| 61.9k/61.9k [00:00<00:00, 191kB/s]

Extracting files...
=== First 5 rows ===
  Country name            Regional indicator  Ladder score  upperwhisker  \
0      Finland                Western Europe         7.741         7.815   
1      Denmark                Western Europe         7.583         7.665   
2      Iceland                Western Europe         7.525         7.618   
3       Sweden                Western Europe         7.344         7.422   
4       Israel  Middle East and North Africa         7.341         7.405   

   lowerwhisker  Log GDP per capita  Social support  Healthy life expectancy  \
0         7.667               1.844           1.572                    0.695   
1         7.500               1.908           1.520                    0.699   
2         7.433               1.881           1.617                    0.718   
3         7.267               1.878           1.501                    0.724   
4         7.277               1.803           1.513                    0.740   

   Freedom to make li

**Q2.** Check for missing values in each column using `.isnull().sum()`. Fill missing values in `Generosity` with the column's mean. Verify the fill worked by checking the null count again.

In [2]:
# Q2: Check, fill, and verify missing values in Generosity

# Step 1: Check missing values across all columns
print("=== Missing values before filling ===")
print(df.isnull().sum())

# Step 2: Fill missing Generosity values with the column mean
generosity_mean = df["Generosity"].mean()
df["Generosity"] = df["Generosity"].fillna(generosity_mean)

# Step 3: Verify the fill worked
print("\n=== Missing values after filling ===")
print(df.isnull().sum())
print(f"\nGenerosity mean used for fill: {generosity_mean:.4f}")

=== Missing values before filling ===
Country name                    0
Regional indicator              0
Ladder score                    0
upperwhisker                    0
lowerwhisker                    0
Log GDP per capita              3
Social support                  3
Healthy life expectancy         3
Freedom to make life choices    3
Generosity                      3
Perceptions of corruption       3
Dystopia + residual             3
dtype: int64

=== Missing values after filling ===
Country name                    0
Regional indicator              0
Ladder score                    0
upperwhisker                    0
lowerwhisker                    0
Log GDP per capita              3
Social support                  3
Healthy life expectancy         3
Freedom to make life choices    3
Generosity                      0
Perceptions of corruption       3
Dystopia + residual             3
dtype: int64

Generosity mean used for fill: 0.1463


### How it works

- `.isnull().sum()` counts missing values per column — run it before and after to confirm the fix.
- `.mean()` computes the average of all non-null values in the column.
- `.fillna(value)` replaces every `NaN` in the column with the given value — here the column mean.
- We reassign the result back to `df["Generosity"]` because `.fillna()` returns a new Series, it does not modify in place.

**Key takeaway:** Filling with the column mean is the simplest strategy for numeric missing values — it keeps the overall average unchanged and avoids losing rows by dropping them.

**Q3.** Drop all rows where `Ladder Score` (formerly `Life Ladder`) is missing. Print how many rows were removed by comparing `.shape` before and after.

In [4]:
# Q3: Drop rows with missing Ladder score and count removed rows

# Step 1: Record shape before dropping
rows_before = df.shape[0]

# Step 2: Drop rows where Ladder score is missing
df = df.dropna(subset=["Ladder score"])

# Step 3: Compare shape after dropping
rows_after = df.shape[0]

print(f"Rows before : {rows_before}")
print(f"Rows after  : {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

Rows before : 143
Rows after  : 143
Rows removed: 0


### How it works

- `df.shape[0]` gives the row count — we save it before and after to measure the difference.
- `.dropna(subset=["Ladder score"])` removes only rows where `Ladder score` is `NaN` — other columns are unaffected.
- Without `subset`, `.dropna()` would drop any row that has a missing value in *any* column, which is too aggressive here.
- We reassign the result to `df` because `.dropna()` returns a new DataFrame, it does not modify in place.

**Key takeaway:** Always use `subset=["col"]` with `.dropna()` to target a specific column — dropping on all columns at once often removes far more rows than intended.

**Q4.** Use `groupby()` to find the **mean Ladder Score** for each `Regional indicator`. Sort the result from highest to lowest and print it.

In [5]:
# Q4: Mean Ladder score by region, sorted descending

regional_happiness = (df.groupby("Regional indicator")["Ladder score"]
                        .mean()
                        .sort_values(ascending=False))

print("=== Mean Happiness Score by Region ===")
print(regional_happiness.round(3))

=== Mean Happiness Score by Region ===
Regional indicator
North America and ANZ                 6.928
Western Europe                        6.842
Central and Eastern Europe            6.171
Latin America and Caribbean           6.143
East Asia                             5.934
Southeast Asia                        5.552
Commonwealth of Independent States    5.538
Middle East and North Africa          5.200
Sub-Saharan Africa                    4.330
South Asia                            3.896
Name: Ladder score, dtype: float64


### How it works

- `.groupby("Regional indicator")` splits the DataFrame into groups — one group per unique region.
- `["Ladder score"]` selects only that column from each group, then `.mean()` computes the average per group.
- `.sort_values(ascending=False)` sorts the resulting Series from highest to lowest mean score.
- `.round(3)` keeps the output clean — happiness scores don't need more than 3 decimal places.

**Key takeaway:** `groupby() → select column → aggregation` is the core pandas pattern for summarizing data by category — think of it as the pandas equivalent of SQL's `GROUP BY`.

**Q5.** Use `groupby()` with `agg()` to compute both the **mean** and **max** of `Log GDP per capita` for each `Regional indicator`. Print the result.

In [6]:
# Q5: Mean and max of Log GDP per capita by region using agg()

gdp_stats = (df.groupby("Regional indicator")["Log GDP per capita"]
               .agg(["mean", "max"])
               .round(3))

print("=== Mean and Max Log GDP per capita by Region ===")
print(gdp_stats)

=== Mean and Max Log GDP per capita by Region ===
                                     mean    max
Regional indicator                              
Central and Eastern Europe          1.638  1.786
Commonwealth of Independent States  1.401  1.642
East Asia                           1.700  1.909
Latin America and Caribbean         1.328  1.702
Middle East and North Africa        1.461  1.983
North America and ANZ               1.861  1.939
South Asia                          1.052  1.361
Southeast Asia                      1.374  2.118
Sub-Saharan Africa                  0.904  1.570
Western Europe                      1.873  2.141


### How it works

- `.agg(["mean", "max"])` applies multiple aggregation functions at once to the selected column.
- The result is a DataFrame where each row is a region and each column is one aggregation (`mean`, `max`).
- `.round(3)` is chained at the end to keep all values to 3 decimal places for readability.

**Key takeaway:** `.agg()` lets you apply multiple functions in one step — instead of writing separate `.mean()` and `.max()` calls, pass them together as a list to get both results in a single clean DataFrame.

**Q6.** Use `pivot_table()` to create a table showing the **average Ladder Score** for each `Regional indicator` per `year`. Print the result with `year` as columns.

In [9]:
# Q6: Pivot table — average key metrics by Regional indicator

pivot = pd.pivot_table(
    df,
    values=["Ladder score", "Log GDP per capita", "Generosity"],  # metrics
    index="Regional indicator",                                    # rows
    aggfunc="mean"                                                 # aggregation
).round(3)

print("=== Average Key Metrics by Region ===")
print(pivot)

=== Average Key Metrics by Region ===
                                    Generosity  Ladder score  \
Regional indicator                                             
Central and Eastern Europe               0.135         6.171   
Commonwealth of Independent States       0.143         5.538   
East Asia                                0.122         5.934   
Latin America and Caribbean              0.107         6.143   
Middle East and North Africa             0.113         5.200   
North America and ANZ                    0.226         6.928   
South Asia                               0.150         3.896   
Southeast Asia                           0.223         5.552   
Sub-Saharan Africa                       0.150         4.330   
Western Europe                           0.171         6.842   

                                    Log GDP per capita  
Regional indicator                                      
Central and Eastern Europe                       1.638  
Commonwealth of Indepe

In [8]:
print(df.columns)

Index(['Country name', 'Regional indicator', 'Ladder score', 'upperwhisker',
       'lowerwhisker', 'Log GDP per capita', 'Social support',
       'Healthy life expectancy', 'Freedom to make life choices', 'Generosity',
       'Perceptions of corruption', 'Dystopia + residual'],
      dtype='str')


**Q7.** Use `apply()` to create a new column `WealthCategory`:
- `"Low"` if `Log GDP per capita` < 8
- `"Medium"` if between 8 and 10
- `"High"` if >= 10
Print `.value_counts()` for the new column.

In [10]:
# Q7: Create WealthCategory column using apply()

# Step 1: Define the categorization function
def categorize_wealth(gdp):
    if gdp < 8:
        return "Low"
    elif gdp <= 10:
        return "Medium"
    else:
        return "High"

# Step 2: Apply the function row by row to Log GDP per capita
df["WealthCategory"] = df["Log GDP per capita"].apply(categorize_wealth)

# Step 3: Print distribution of each category
print("=== WealthCategory value counts ===")
print(df["WealthCategory"].value_counts())

=== WealthCategory value counts ===
WealthCategory
Low     140
High      3
Name: count, dtype: int64


### Q8. Split the DataFrame into two smaller DataFrames:

- `df_left` — keep columns Country name, Ladder score, and Generosity
- `df_right` — keep columns Country name, Log GDP per capita, and WealthCategory

Use `pd.merge()` to join them on Country name. Print the first 5 rows of the merged result.

In [11]:
# Q8: Create two smaller DataFrames and merge them

# Step 1: Create df_low — Low wealth countries with Ladder score
df_low = df[df["WealthCategory"] == "Low"][["Country name", "Ladder score"]].reset_index(drop=True)

# Step 2: Create df_high — High wealth countries with Ladder score
df_high = df[df["WealthCategory"] == "High"][["Country name", "Ladder score"]].reset_index(drop=True)

# Step 3: Split df into two halves to simulate a merge scenario
df_left = df[["Country name", "Ladder score", "Generosity"]].copy()
df_right = df[["Country name", "Log GDP per capita", "WealthCategory"]].copy()

# Step 4: Merge on Country name
merged = pd.merge(df_left, df_right, on="Country name", suffixes=("_left", "_right"))

print("=== Merged DataFrame — first 5 rows ===")
print(merged.head())

=== Merged DataFrame — first 5 rows ===
  Country name  Ladder score  Generosity  Log GDP per capita WealthCategory
0      Finland         7.741       0.142               1.844            Low
1      Denmark         7.583       0.204               1.908            Low
2      Iceland         7.525       0.258               1.881            Low
3       Sweden         7.344       0.221               1.878            Low
4       Israel         7.341       0.153               1.803            Low


### How it works

- `pd.merge()` joins two DataFrames on a shared column — here `"Country name"` is the common key.
- `suffixes=("_left", "_right")` renames duplicate columns so you can tell which DataFrame they came from.
- This simulates the intended exercise: splitting data into two sources and rejoining them.
- The default `how="inner"` keeps only rows where `Country name` exists in both DataFrames.

**Key takeaway:** `pd.merge()` works like a SQL JOIN — always specify `on=` (the shared key column) and `suffixes=` when both DataFrames share column names beyond the key, to avoid ambiguous column names.

**Q9.**  From the merged DataFrame in Q8, add a new column ScoreGap = Ladder score - Generosity. Sort by Ladder score in descending order and print the top 5 countries with their Country name, Ladder score, Generosity, and ScoreGap columns..

In [12]:
# Q9: Add derived column and find top 5 countries by Ladder score

# Step 1: Add a ScoreGap column — difference between Ladder score and Generosity
merged["ScoreGap"] = merged["Ladder score"] - merged["Generosity"]

# Step 2: Sort by Ladder score descending and print top 5
top5 = merged.sort_values("Ladder score", ascending=False).head()

print("=== Top 5 Countries by Ladder Score ===")
print(top5[["Country name", "Ladder score", "Generosity", "ScoreGap"]])

=== Top 5 Countries by Ladder Score ===
  Country name  Ladder score  Generosity  ScoreGap
0      Finland         7.741       0.142     7.599
1      Denmark         7.583       0.204     7.379
2      Iceland         7.525       0.258     7.267
3       Sweden         7.344       0.221     7.123
4       Israel         7.341       0.153     7.188


**Q10.** Use `concat()` to stack `df_left` and `df_right` (from Q8) vertically into a single DataFrame. Reset the index and print its shape and first 5 rows.

In [13]:
# Q10: Vertically stack df_left and df_right using concat()

# Step 1: Concatenate vertically (axis=0 means stack rows)
df_concat = pd.concat([df_left, df_right], axis=0)

# Step 2: Reset index so row numbers are continuous
df_concat = df_concat.reset_index(drop=True)

# Step 3: Print shape and first 5 rows
print(f"Shape after concat: {df_concat.shape}")
print("\n=== First 5 rows ===")
print(df_concat.head())

Shape after concat: (286, 5)

=== First 5 rows ===
  Country name  Ladder score  Generosity  Log GDP per capita WealthCategory
0      Finland         7.741       0.142                 NaN            NaN
1      Denmark         7.583       0.204                 NaN            NaN
2      Iceland         7.525       0.258                 NaN            NaN
3       Sweden         7.344       0.221                 NaN            NaN
4       Israel         7.341       0.153                 NaN            NaN


### How it works

- `pd.concat([df1, df2], axis=0)` stacks two DataFrames vertically — rows of `df_right` are appended below rows of `df_left`.
- Columns that exist in one DataFrame but not the other will appear with `NaN` for the rows that lack them.
- `reset_index(drop=True)` renumbers rows from 0 continuously — without it, index values from both DataFrames are preserved and may repeat.
- `drop=True` discards the old index instead of adding it as a column.

**Key takeaway:** `concat()` with `axis=0` stacks rows — mismatched columns are kept and filled with `NaN`, so always check `.shape` and `.head()` after to confirm the result looks as expected.

**Q11.** Use `map()` to create a new column `IsHappy` that is `True` if `Ladder Score >= 6.0`, else `False`. Print the count of happy vs. not-happy country-years.

In [14]:
# Q11: Create IsHappy column using map()

# Step 1: map() works on a Series — pass a lambda function
df["IsHappy"] = df["Ladder score"].map(lambda x: x >= 6.0)

# Step 2: Print count of True vs False
print("=== Happy vs Not-Happy Countries ===")
print(df["IsHappy"].value_counts())

=== Happy vs Not-Happy Countries ===
IsHappy
False    87
True     56
Name: count, dtype: int64


### How it works

- `.map()` applies a function to every value in a Series — here a `lambda` that returns `True` or `False`.
- `lambda x: x >= 6.0` is a one-line anonymous function — it takes one value `x` and returns a boolean.
- `.value_counts()` counts how many `True` and `False` values exist in the new column.
- The difference between `.map()` and `.apply()`: `.map()` is for element-wise operations on a **Series**; `.apply()` can work on both Series and DataFrames.

**Key takeaway:** `.map()` is the cleanest way to transform a single column value by value — pass a `lambda` for simple one-line logic, or a named function for anything more complex.

**Q12.** Find the top 10 countries with the highest **average Ladder Score** across all years. Use `groupby()` and `.mean()`, then sort and print the top 10.

In [15]:
# Q12: Top 10 happiest countries by Ladder score

top10_countries = (df[["Country name", "Ladder score"]]
                     .sort_values("Ladder score", ascending=False)
                     .head(10)
                     .reset_index(drop=True))

print("=== Top 10 Happiest Countries ===")
print(top10_countries)

=== Top 10 Happiest Countries ===
  Country name  Ladder score
0      Finland         7.741
1      Denmark         7.583
2      Iceland         7.525
3       Sweden         7.344
4       Israel         7.341
5  Netherlands         7.319
6       Norway         7.302
7   Luxembourg         7.122
8  Switzerland         7.060
9    Australia         7.057


### How it works

- Since the dataset has only one year, `groupby()` + `.mean()` would return the same value as the original score — so sorting directly is the honest equivalent.
- `.sort_values("Ladder score", ascending=False)` ranks countries from happiest to least happy.
- `.head(10)` keeps only the top 10 rows.
- `reset_index(drop=True)` gives clean row numbers 1–10 instead of the original DataFrame index.

**Key takeaway:** `groupby().mean()` is meaningful only when multiple rows exist per group — with a single-year snapshot, sorting and slicing with `.head(N)` is the correct and equivalent approach.

**Q13.** Filter the dataset to the most recent year available. Among those rows, print the country with the highest `Freedom to make life choices` and the country with the lowest `Perceptions of corruption`.

In [16]:
# Q13: Most recent year filter, then find extremes in two columns

# Step 1: This is a single-year dataset — all rows are the most recent year
# Confirm by checking unique years if a year column existed
# We treat the full df as the most recent year

# Step 2: Country with highest Freedom to make life choices
most_free_idx = df["Freedom to make life choices"].idxmax()
most_free = df.loc[most_free_idx, ["Country name", "Freedom to make life choices"]]

print("=== Country with highest Freedom to make life choices ===")
print(f"Country : {most_free['Country name']}")
print(f"Score   : {most_free['Freedom to make life choices']:.3f}")

# Step 3: Country with lowest Perceptions of corruption
least_corrupt_idx = df["Perceptions of corruption"].idxmin()
least_corrupt = df.loc[least_corrupt_idx, ["Country name", "Perceptions of corruption"]]

print("\n=== Country with lowest Perceptions of corruption ===")
print(f"Country : {least_corrupt['Country name']}")
print(f"Score   : {least_corrupt['Perceptions of corruption']:.3f}")

=== Country with highest Freedom to make life choices ===
Country : Cambodia
Score   : 0.863

=== Country with lowest Perceptions of corruption ===
Country : Bosnia and Herzegovina
Score   : 0.000


### How it works

- Since this is a single-year dataset, all rows already represent the most recent year — no filtering needed.
- `.idxmax()` returns the index label of the row with the highest value; `.idxmin()` returns the lowest.
- `.loc[idx, ["col1", "col2"]]` fetches only the relevant columns for that specific row.
- Individual values are accessed from the resulting Series using `row["column name"]`.

**Key takeaway:** `.idxmax()` and `.idxmin()` are the cleanest way to locate extreme values — combine them with `.loc[]` to retrieve any additional columns from that same row without writing a separate filter.

**Q14.** Use `.apply()` with `axis=1` to create a column `WellbeingIndex = (Ladder Score + Social support + Freedom to make life choices) / 3`. Print the top 5 countries by this index for the most recent year.

In [18]:
# Q14: Create WellbeingIndex using apply() with axis=1

# Step 1: Define function that operates on a single row
def wellbeing_index(row):
    return (row["Ladder score"] + row["Social support"] + row["Freedom to make life choices"]) / 3

# Step 2: Apply row-wise (axis=1 means each row is passed to the function)
df["WellbeingIndex"] = df.apply(wellbeing_index, axis=1)

# Step 3: Print top 5 countries by WellbeingIndex
top5 = (df[["Country name", "Ladder score", "Social support", "Freedom to make life choices", "WellbeingIndex"]]
          .sort_values("WellbeingIndex", ascending=False)
          .head()
          .reset_index(drop=True))

print("=== Top 5 Countries by WellbeingIndex ===")
print(top5.round(3))

=== Top 5 Countries by WellbeingIndex ===
  Country name  Ladder score  Social support  Freedom to make life choices  \
0      Finland         7.741           1.572                         0.859   
1      Iceland         7.525           1.617                         0.819   
2      Denmark         7.583           1.520                         0.823   
3       Sweden         7.344           1.501                         0.838   
4       Norway         7.302           1.517                         0.835   

   WellbeingIndex  
0           3.391  
1           3.320  
2           3.309  
3           3.228  
4           3.218  


### How it works

- `axis=1` tells `.apply()` to pass one **row at a time** to the function — each `row` is a Series with column names as its index.
- Inside the function, `row["column name"]` accesses individual values from that row.
- This is different from Q11 where `.map()` received one **value** at a time from a single column.
- `.round(3)` keeps all printed values to 3 decimal places for a clean output.

**Key takeaway:** Use `.apply(func, axis=1)` when your calculation needs values from **multiple columns in the same row** — `axis=1` is the key difference from the default `axis=0` which passes one column at a time.

**Q15.** Group by `Regional indicator` and count the number of unique countries in each region. Sort by count descending and print. *(Hint: use `.nunique()` on the `Country` column after groupby.)*